# 01. Exploración Inicial de Datos (EDA)

**Materia:** Programación Análisis de Datos (PAD)  
**Dataset:** *Financial Transactions Dataset: Analytics* (Kaggle)  
**Fuente de datos local:** `data/proyecto.sqlite`

## Objetivo del notebook
Este notebook proporciona una plantilla reproducible para:
1. Conectarse a la base de datos local `data/proyecto.sqlite` generada con `python src/crear_db.py`.
2. Listar las tablas disponibles, su volumen de registros y su esquema.
3. Realizar una inspección básica de calidad de datos (tipos, valores nulos y estadísticas descriptivas) usando `pandas`.
4. Generar visualizaciones exploratorias iniciales con `matplotlib` y `seaborn`.

## 1. Configuración e importación de librerías

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Configuración visual global para gráficos y tablas
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 2. Conexión a `data/proyecto.sqlite`

Localizamos automáticamente la raíz del repositorio para que el notebook funcione tanto si Jupyter se inicia desde la raíz del proyecto como desde la carpeta `notebooks/`.

In [ ]:
def localizar_base_datos() -> Path:
    """Busca `data/proyecto.sqlite` desde el directorio actual o su directorio padre."""
    candidatos = [
        Path.cwd() / "data" / "proyecto.sqlite",
        Path.cwd().parent / "data" / "proyecto.sqlite",
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    raise FileNotFoundError(
        "No se encontró 'data/proyecto.sqlite'. "
        "Coloca los archivos .csv de Kaggle en 'data/' y ejecuta: python src/crear_db.py"
    )


DB_PATH = localizar_base_datos()
conn = sqlite3.connect(DB_PATH)
print(f"Conexión establecida con: {DB_PATH.resolve()}")
print(f"Tamaño de la base de datos: {DB_PATH.stat().st_size / (1024 * 1024):.2f} MB")

## 3. Listado de tablas disponibles y resumen de volumen

In [ ]:
tablas_df = pd.read_sql_query(
    "SELECT name AS tabla FROM sqlite_master WHERE type = 'table' ORDER BY name;",
    conn,
)

resumen_tablas = []
for nombre_tabla in tablas_df["tabla"]:
    num_filas = pd.read_sql_query(
        f'SELECT COUNT(*) AS n FROM "{nombre_tabla}";', conn
    )["n"].iloc[0]
    esquema = pd.read_sql_query(f'PRAGMA table_info("{nombre_tabla}");', conn)
    resumen_tablas.append(
        {
            "tabla": nombre_tabla,
            "num_filas": int(num_filas),
            "num_columnas": len(esquema),
            "columnas": ", ".join(esquema["name"].tolist()),
        }
    )

df_resumen = pd.DataFrame(
    resumen_tablas,
    columns=["tabla", "num_filas", "num_columnas", "columnas"],
)
display(df_resumen)

## 4. Inspección básica con `pandas`

Para tablas muy grandes (como `transactions_data`, con millones de registros), cargamos una muestra controlada mediante `LIMIT` para realizar una inspección ágil sin saturar la memoria RAM.

In [ ]:
MUESTRA_MAX = 50_000
muestras: dict[str, pd.DataFrame] = {}

for nombre_tabla in df_resumen["tabla"]:
    df_muestra = pd.read_sql_query(
        f'SELECT * FROM "{nombre_tabla}" LIMIT {MUESTRA_MAX};', conn
    )
    muestras[nombre_tabla] = df_muestra

    print("=" * 75)
    print(f"TABLA: {nombre_tabla} | Muestra cargada: {len(df_muestra):,} filas x {df_muestra.shape[1]} columnas")
    print("=" * 75)

    display(df_muestra.head(5))

    calidad = pd.DataFrame(
        {
            "tipo_dato": df_muestra.dtypes.astype(str),
            "nulos": df_muestra.isna().sum(),
            "pct_nulos": (df_muestra.isna().mean() * 100).round(2),
            "valores_unicos": df_muestra.nunique(dropna=True),
        }
    )
    print("\nResumen de tipos, valores nulos y cardinalidad (en la muestra):")
    display(calidad)

    print("\nEstadísticas descriptivas:")
    display(df_muestra.describe(include="all").T)

## 5. Visualizaciones exploratorias iniciales (`matplotlib` y `seaborn`)

Primero visualizamos el volumen de registros almacenado en cada tabla de `data/proyecto.sqlite`.

In [ ]:
if not df_resumen.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    datos_ordenados = df_resumen.sort_values("num_filas", ascending=True)
    sns.barplot(
        data=datos_ordenados,
        x="num_filas",
        y="tabla",
        hue="tabla",
        palette="viridis",
        legend=False,
        ax=ax,
    )
    ax.set_title("Volumen de registros por tabla en proyecto.sqlite")
    ax.set_xlabel("Número de registros")
    ax.set_ylabel("Tabla")

    for contenedor in ax.containers:
        ax.bar_label(contenedor, fmt="{:,.0f}", padding=4)

    plt.tight_layout()
    plt.show()

### 5.1 Distribución de montos y canales de pago (`transactions_data`)

En el dataset original, las columnas monetarias (como `amount`, `credit_limit` o `yearly_income`) suelen incluir el símbolo `$`. Definimos una función auxiliar para convertirlas a formato numérico de forma segura antes de graficar.

In [ ]:
def limpiar_moneda(serie: pd.Series) -> pd.Series:
    """Convierte valores monetarios con '$' y comas a tipo numérico float."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie.astype(float)
    limpia = serie.astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(limpia, errors="coerce")


if "transactions_data" in muestras:
    df_tx = muestras["transactions_data"].copy()

    if "amount" in df_tx.columns:
        df_tx["amount_num"] = limpiar_moneda(df_tx["amount"])
        montos_validos = df_tx["amount_num"].dropna()

        if not montos_validos.empty:
            # Acotamos visualmente entre los percentiles 1% y 99% para facilitar la lectura
            p01, p99 = montos_validos.quantile([0.01, 0.99])
            montos_recortados = montos_validos[(montos_validos >= p01) & (montos_validos <= p99)]

            fig, axes = plt.subplots(1, 2, figsize=(14, 5))

            sns.histplot(montos_recortados, bins=40, kde=True, color="#2b8cbe", ax=axes[0])
            axes[0].set_title("Distribución de montos de transacción (P1–P99, muestra)")
            axes[0].set_xlabel("Monto (USD)")
            axes[0].set_ylabel("Frecuencia")

            if "use_chip" in df_tx.columns:
                conteo_canal = df_tx["use_chip"].fillna("Desconocido").value_counts().reset_index()
                conteo_canal.columns = ["use_chip", "conteo"]
                sns.barplot(
                    data=conteo_canal,
                    x="use_chip",
                    y="conteo",
                    hue="use_chip",
                    palette="crest",
                    legend=False,
                    ax=axes[1],
                )
                axes[1].set_title("Transacciones por método de captura (use_chip)")
                axes[1].set_xlabel("Canal de pago")
                axes[1].set_ylabel("Cantidad (muestra)")
                axes[1].tick_params(axis="x", rotation=15)
                for contenedor in axes[1].containers:
                    axes[1].bar_label(contenedor, fmt="{:,.0f}", padding=3)
            else:
                axes[1].axis("off")

            plt.tight_layout()
            plt.show()
else:
    print("La tabla 'transactions_data' no está presente; omitiendo gráficos de transacciones.")

### 5.2 Perfil de tarjetas (`cards_data`) y clientes (`users_data`)

In [ ]:
if "cards_data" in muestras:
    df_cards = muestras["cards_data"].copy()
    if {"card_brand", "card_type"}.issubset(df_cards.columns):
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.countplot(data=df_cards, x="card_brand", hue="card_type", palette="Set2", ax=ax)
        ax.set_title("Distribución de tarjetas por marca y tipo")
        ax.set_xlabel("Marca de tarjeta")
        ax.set_ylabel("Cantidad de tarjetas")
        ax.legend(title="Tipo de tarjeta")
        plt.tight_layout()
        plt.show()

if "users_data" in muestras:
    df_users = muestras["users_data"].copy()
    for col_moneda in ("yearly_income", "per_capita_income", "total_debt"):
        if col_moneda in df_users.columns:
            df_users[col_moneda] = limpiar_moneda(df_users[col_moneda])

    cols_corr = [
        c
        for c in ("current_age", "yearly_income", "total_debt", "credit_score", "num_credit_cards")
        if c in df_users.columns
    ]
    if len(cols_corr) >= 2:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        if "credit_score" in df_users.columns:
            sns.histplot(df_users["credit_score"].dropna(), bins=30, kde=True, color="#31a354", ax=axes[0])
            axes[0].set_title("Distribución del puntaje crediticio (credit_score)")
            axes[0].set_xlabel("Puntaje crediticio")
            axes[0].set_ylabel("Número de clientes")
        else:
            axes[0].axis("off")

        matriz_corr = df_users[cols_corr].corr(numeric_only=True)
        sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
        axes[1].set_title("Matriz de correlación de variables financieras (users_data)")

        plt.tight_layout()
        plt.show()

## 6. Cierre de la conexión y próximos pasos

Siempre cerramos la conexión a SQLite al terminar el análisis para liberar el archivo local.

In [ ]:
conn.close()
print("Conexión cerrada correctamente.")

### Ideas para continuar el análisis
- Ejecutar las consultas analíticas y `JOIN` de ejemplo en [`sql/consultas.sql`](../sql/consultas.sql).
- Crear índices en `data/proyecto.sqlite` (`python src/crear_db.py --crear-indices`) para acelerar cruces entre transacciones, tarjetas y clientes.
- Incorporar los archivos JSON complementarios del dataset (`mcc_codes.json` y `train_fraud_labels.json`) para enriquecer categorías comerciales y analizar patrones de fraude.